# Week 6 · Optimisers: Step Size, Conditioning, Momentum and Adam

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/weisongwen/AAE4011-S22627/blob/main/Lecture%20Notes/Week%206/w6_optimisers.ipynb)

AAE4011 Artificial Intelligence in Unmanned Autonomous Systems · Semester 2, 2026/27

Self-study for Section 3 of the slides. Everything runs in a second; no data files are needed.

**Questions**
- (a) On the quadratic of Section 3, which step size α is the fastest, and above which α does gradient descent diverge? Explain with the eigenvalues of $\mathbf{H}$.
- (b) How many iterations does gradient descent need at condition number κ = 100? At κ = 1000?
- (c) Momentum gets close to the minimum sooner but overshoots. Why? What does Nesterov change?
- (d) In its first step, Adam moves x and y by the same amount, although the y gradient is 100 times larger. Why?

## Part 1 · Gradient descent on the quadratic of Section 3

$$f(\boldsymbol\theta) = \tfrac12(\boldsymbol\theta - \boldsymbol\theta^*)^{\mathsf T}\mathbf{H}(\boldsymbol\theta - \boldsymbol\theta^*), \qquad \mathbf{H} = \begin{bmatrix}5 & 3\\ 3 & 5\end{bmatrix}, \qquad \boldsymbol\theta^* = (2, 1), \qquad \nabla f = \mathbf{H}(\boldsymbol\theta - \boldsymbol\theta^*).$$

Start at $\boldsymbol\theta_0 = (0, 0)$, where $f_0 = 18.5$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

H = np.array([[5.0, 3.0], [3.0, 5.0]])
t_star = np.array([2.0, 1.0])
f = lambda t: 0.5 * (t - t_star) @ H @ (t - t_star)
grad = lambda t: H @ (t - t_star)

def gd(alpha, steps, t0=(0.0, 0.0)):
    path = [np.array(t0, float)]
    for _ in range(steps):
        path.append(path[-1] - alpha * grad(path[-1]))
    return np.array(path)

for k, t in enumerate(gd(0.2, 2)):
    print(f"step {k}: theta = ({t[0]:.2f}, {t[1]:.2f}),  f = {f(t):.4f}")

Along each eigenvector of $\mathbf{H}$ the error is multiplied by $1 - \alpha\lambda$ every step. With $\lambda$ = 2 and 8, the slowest factor decides the speed.

In [ ]:
lam = np.linalg.eigvalsh(H)
print("eigenvalues", lam, " alpha_max = 2/lam_max =", 2 / lam.max(), " alpha* = 2/(lam_min + lam_max) =",
      2 / lam.sum())
for a in (0.05, 0.1, 0.2, 0.24, 0.25, 0.26):
    rho = np.max(np.abs(1 - a * lam))
    verdict = "converges" if rho < 1 else ("bounces for ever" if np.isclose(rho, 1) else "diverges")
    print(f"alpha {a:4.2f}: slowest factor {rho:.2f}, f after 30 steps {f(gd(a, 30)[-1]):9.3g}  -> {verdict}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
X, Y = np.meshgrid(np.linspace(-1, 5, 200), np.linspace(-2, 4, 200))
Z = 0.5 * (5 * (X - 2) ** 2 + 6 * (X - 2) * (Y - 1) + 5 * (Y - 1) ** 2)
ax[0].contour(X, Y, Z, levels=[0.5, 2, 5, 10, 18.5, 30], colors="0.7")
for a, c in ((0.1, "C0"), (0.2, "C2"), (0.24, "C1")):
    p = gd(a, 15)
    ax[0].plot(p[:, 0], p[:, 1], "o-", ms=3, color=c, label=f"alpha {a}")
    ax[1].semilogy([f(t) + 1e-16 for t in gd(a, 40)], color=c, label=f"alpha {a}")
ax[0].plot(*t_star, "k*", ms=12); ax[0].set_aspect("equal"); ax[0].legend(); ax[0].set_title("Paths")
ax[1].set_xlabel("step"); ax[1].set_ylabel("f"); ax[1].legend(); ax[1].set_title("Loss curve")
plt.tight_layout(); plt.show()

## Part 2 · Conditioning → question (b)

$\mathbf{H} = \operatorname{diag}(1, \kappa)$ with the best fixed step $\alpha^* = 2/(1 + \kappa)$: both directions shrink by $(\kappa - 1)/(\kappa + 1)$ per step. Count the steps until the error is a million times smaller.

In [ ]:
for kappa in (4, 25, 100, 1000):
    Hk = np.diag([1.0, kappa])
    a = 2 / (1 + kappa)
    t = np.array([1.0, 1.0])
    n = 0
    while np.linalg.norm(t) > 1e-6 * np.sqrt(2):
        t = t - a * Hk @ t
        n += 1
    print(f"kappa {kappa:4d}: rate {(kappa - 1) / (kappa + 1):.3f}, {n:4d} iterations")

## Part 3 · Momentum and Nesterov → question (c)

$f(\theta) = \theta^2/2$ from $\theta_0 = 10$, $\alpha = 0.1$, $\beta = 0.9$, in the PyTorch form:

- momentum: $v \leftarrow \beta v + g$, $\theta \leftarrow \theta - \alpha v$;
- Nesterov: $v \leftarrow \beta v + g$, $\theta \leftarrow \theta - \alpha(g + \beta v)$.

In [ ]:
def run_1d(kind, steps=60, alpha=0.1, beta=0.9, t=10.0):
    v, out = 0.0, [t]
    for _ in range(steps):
        g = t
        if kind == "gd":
            t = t - alpha * g
        else:
            v = beta * v + g
            t = t - alpha * (v if kind == "momentum" else g + beta * v)
        out.append(t)
    return np.array(out)

runs = {k: run_1d(k) for k in ("gd", "momentum", "nesterov")}
print("step      GD   momentum   Nesterov")
for k in range(9):
    print(f"{k:4d} {runs['gd'][k]:7.3f} {runs['momentum'][k]:10.3f} {runs['nesterov'][k]:10.3f}")
for kind, r in runs.items():
    print(f"{kind:9s}: lowest theta {r.min():6.2f}, |theta| after 60 steps {abs(r[-1]):.1e}")

plt.figure(figsize=(7, 4))
for kind, r in runs.items():
    plt.plot(r, label=kind)
plt.axhline(0, color="0.6", lw=0.8); plt.xlabel("step"); plt.ylabel("theta"); plt.legend(); plt.show()

## Part 4 · Adam on a badly scaled bowl → question (d)

$f = (x^2 + 100y^2)/2$ from (1, 1). Gradient descent must keep $\alpha < 2/100$ for the steep $y$ direction, so it crawls along $x$. Adam divides each coordinate by its own running gradient size.

In [ ]:
gb = lambda t: np.array([t[0], 100 * t[1]])

def adam(alpha=0.1, steps=100, b1=0.9, b2=0.999, eps=1e-8, t0=(1.0, 1.0)):
    t, m, s, path = np.array(t0), np.zeros(2), np.zeros(2), [np.array(t0)]
    for k in range(1, steps + 1):
        g = gb(t)
        m = b1 * m + (1 - b1) * g
        s = b2 * s + (1 - b2) * g ** 2
        m_hat, s_hat = m / (1 - b1 ** k), s / (1 - b2 ** k)
        t = t - alpha * m_hat / (np.sqrt(s_hat) + eps)
        path.append(t)
    return np.array(path)

pa = adam()
pg = [np.array([1.0, 1.0])]
for _ in range(100):
    pg.append(pg[-1] - 0.019 * gb(pg[-1]))
pg = np.array(pg)
print(f"Adam step 1: ({pa[1, 0]:.4f}, {pa[1, 1]:.4f});  step 2: ({pa[2, 0]:.4f}, {pa[2, 1]:.4f})")
print(f"GD, alpha 0.019, step 1: ({pg[1, 0]:.3f}, {pg[1, 1]:.3f})")
print(f"after 100 steps, distance to the minimum: Adam {np.linalg.norm(pa[-1]):.3f}, "
      f"GD {np.linalg.norm(pg[-1]):.3f}")

plt.figure(figsize=(7, 4))
plt.plot(pg[:, 0], pg[:, 1], ".-", ms=3, label="GD, alpha 0.019")
plt.plot(pa[:, 0], pa[:, 1], ".-", ms=3, label="Adam, alpha 0.1")
plt.plot(0, 0, "k*", ms=12); plt.legend(); plt.xlabel("x"); plt.ylabel("y"); plt.show()

**Why the bias correction?** $\mathbf{m}$ and $\mathbf{s}$ start at zero, so in the first steps they are too small. With a constant gradient $g$, $m_t = (1 - 0.9^t)\,g$ and $s_t = (1 - 0.999^t)\,g^2$.

In [ ]:
for t in (1, 2, 10, 100, 1000):
    raw = (1 - 0.9 ** t) / np.sqrt(1 - 0.999 ** t)
    print(f"t = {t:4d}: step without correction {raw:5.2f} alpha, with correction 1.00 alpha")

---
## Check your answers (try first!)

- (a) Gradient descent multiplies the error by $1 - 2\alpha$ and $1 - 8\alpha$ along the two eigenvectors. The fastest fixed step is $\alpha^* = 2/(2 + 8) = 0.2$, where both factors have size 0.6. Above $\alpha = 2/8 = 0.25$ the factor $1 - 8\alpha$ is below −1 and the steep direction diverges; at exactly 0.25 it bounces for ever.
- (b) 691 iterations at κ = 100 and 6908 at κ = 1000 (28 at κ = 4). The count grows about like κ: badly scaled problems need scaling, Newton/Gauss–Newton or an adaptive method.
- (c) The velocity $v$ adds up past gradients, so steps grow while the gradient keeps its sign, and $\theta$ is still moving fast when it reaches 0: it overshoots to −6.04. Nesterov adds the gradient once more on top of the velocity; on this bowl it is closer sooner (0.94 against 2.27 at step 4) and overshoots less (−3.94).
- (d) At $t = 1$, $\hat m = g$ and $\hat s = g^2$, so the step is $\alpha\, g/|g| = \alpha$ in every coordinate, whatever the size of $g$: both coordinates move by 0.1. Without the bias correction the first step would be 3.16 α.